# Import libraries and initialising project

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import inception_v3, Inception_V3_Weights
import time
import itertools
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
from tqdm import tqdm
from PIL import Image, ImageOps

sns.set_theme()


NUM_EPOCHS = 15
NUM_CLASSES = 3
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_dir = os.path.join("datasets/train/")
test_dir = os.path.join("datasets/test/")
val_dir = os.path.join("datasets/valid/")
best_global_acc = 0.0
best_params = None

# Set the parameters to test
param_grid = {
    'lr': [0.02, 0.002, 0.0002],
    'weight_decay': [1e-4, 1e-5],
    'batch_size': [32, 64, 128]
}

### Load functions used

In [ ]:
def train_model(model, loader, criterion, optimiser, device):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    with tqdm(loader, desc='Training', leave=False) as pbar:
        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device)

            optimiser.zero_grad(set_to_none=True)

            # Forward pass
            outputs = model(inputs)
            loss1 = criterion(outputs.logits, labels)
            loss2 = criterion(outputs.aux_logits, labels)
            loss = loss1 + 0.4 * loss2

            # Backward pass and optimisation
            loss.backward()
            optimiser.step()

            # Track metrics
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.logits.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            # Update progress bar
            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100. * correct / total:.2f}%'
            })

    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc


def validate_model(model, loader, criterion, device):
    """validate the model"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad(), tqdm(loader, desc='Evaluation', leave=False) as pbar:
        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device)

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Track metrics
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            # Update progress bar
            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100. * correct / total:.2f}%'
            })

    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc


# splits and generates dataloaders for each data splits
def data_generator(parent_directory: str, transform: transforms.Compose, batch_size: int, subset: str):
    if subset not in ["Train", "Val", "Test"]:
        raise ValueError('Subset must be one of "Train", "Val", "Test"')

    dataset = datasets.ImageFolder(
            parent_directory,
            transform=transform
        )
    if subset == "Train":
        train_generator = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=True
        )
        return train_generator

    elif subset in ["Test", "val"]:
        generator = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=False
        )
        return generator

    else:
        return None


# resizes val and test images, and augment training images.
def image_gen_w_aug(train_parent_directory, test_parent_directory, val_parent_directory, batch_size: int):
    train_batch_size = batch_size
    val_batch_size = batch_size*2
    test_batch_size = batch_size*2

    # normalise images using standard ImageNet scaling values
    normalisation = transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )

    # Image resizing and augmentation (training only)
    train_transform = transforms.Compose([
        transforms.Resize((299, 299)),
        transforms.RandomRotation(30),
        transforms.RandomAffine(
            degrees=0,
            translate=(0.1, 0.1),
            scale=(0.8, 1.2)
        ),
        transforms.ToTensor(),
        normalisation
    ])

    test_transform = transforms.Compose([
        transforms.Resize((299, 299)),
        transforms.ToTensor(),
        normalisation
    ])

    # train, val, test generators
    train_generator = data_generator(train_parent_directory, train_transform, train_batch_size, subset="Train")
    val_generator = data_generator(val_parent_directory, test_transform, val_batch_size, subset="Val")
    test_generator = data_generator(test_parent_directory, test_transform, test_batch_size, subset="Test")

    return train_generator, val_generator, test_generator


def build_model(pre_trained_model, num_classes: int):
    pre_trained_model.fc = nn.Sequential(
        nn.Linear(pre_trained_model.fc.in_features, 512),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512, num_classes)
    )

    return pre_trained_model


# isle_of_gods (2022) Answer to: 'early stopping in PyTorch'. Stack Overflow.
# Available at: https://stackoverflow.com/a/73704579 (Accessed: 17 July 2026).
class EarlyStopper:
    def __init__(self, patience=1, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.min_validation_loss = float('inf')

    def early_stop(self, validation_loss):
        if validation_loss < self.min_validation_loss:
            self.min_validation_loss = validation_loss
            self.counter = 0
        elif validation_loss > (self.min_validation_loss + self.min_delta):
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False

def import_and_predict(image_data, model):
    size = (299, 299)
    image = ImageOps.fit(image_data, size, Image.Resampling.LANCZOS)  # prepare image with antialiasing
    image = image.convert('RGB')  # convert image to RGB, Red Green Blue format
    image = np.asarray(image)  # convert image into array
    image = (image.astype(np.float32) / 255.0)  # create image array matrix
    image = np.transpose(image, (2, 0, 1))  # HWC -> CHW
    img_reshape = torch.from_numpy(image).unsqueeze(0)  # add batch dimension

    model.eval()
    with torch.no_grad():
        prediction = model(img_reshape)
        prediction = prediction.numpy()

    return prediction

### Training model and finding best hyperparameters

In [ ]:
# Generate all unique combinations of hyperparameters
keys, values = zip(*param_grid.items())
experiments = [dict(zip(keys, v)) for v in itertools.product(*values)]

# training and validation
weights = Inception_V3_Weights.DEFAULT
criterion = nn.CrossEntropyLoss()
print("Starting training...\n")
start_time = time.time()

for idx, params in enumerate(experiments):
    print(f"\n================ Running Experiment {idx + 1}/{len(experiments)} ================")
    print(f"Parameters: {params}")

    # Re-generate data loaders if batch_size is a hyperparameter
    train_generator, val_generator, test_generator = image_gen_w_aug(train_dir, test_dir, val_dir, params['batch_size'])

    # Re-initialise model architecture
    pre_trained_model = inception_v3(weights=weights, aux_logits=True)
    for param in pre_trained_model.parameters():
        param.requires_grad = False

    model = build_model(pre_trained_model, NUM_CLASSES)
    model.to(device)

    optimiser = optim.Adam(
        model.parameters(),
        lr=params['lr'],
        weight_decay=params['weight_decay']
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimiser, mode='min', factor=0.5, patience=3)
    early_stopper = EarlyStopper(patience=5, min_delta=0.02)
    model_history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }
    best_val_acc = 0.0

    # 3. Inner training loop (Your original code)
    for epoch in range(NUM_EPOCHS):
        train_loss, train_acc = train_model(model, train_generator, criterion, optimiser, device)
        model_history["train_loss"].append(train_loss)
        model_history["train_acc"].append(train_acc)

        val_loss, val_acc = validate_model(model, val_generator, criterion, device)
        model_history["val_loss"].append(val_loss)
        model_history["val_acc"].append(val_acc)

        scheduler.step(val_loss)

        if val_acc > best_val_acc:
            best_val_acc = val_acc

        if early_stopper.early_stop(val_loss):
            print(f"Early stopping triggered at epoch {epoch + 1}")
            break

    print(f"Experiment {idx + 1} finished. Best Val Acc: {best_val_acc:.2f}%")

    # Keep track of the absolute best configuration
    if best_val_acc > best_global_acc:
        best_global_acc = best_val_acc
        best_params = params
        # Save the absolute best model overall
        torch.save(model.state_dict(), "./models/best_model.pth")

print(f"\nBest Accuracy: {best_global_acc:.2f}%")
print(f"Best Hyperparameters: {best_params}")

training_time = time.time() - start_time
print("=" * 70)
print(f"Training completed in {training_time / 60:.2f} minutes")
print("=" * 70)

### Validate model on test set

In [ ]:
# Load best model
model.load_state_dict(torch.load("./models/best_model.pth"))
print("Best model loaded!")

# Validate on test set
test_loss, test_acc = validate_model(model, test_generator, criterion, device)
print("\nTest metrics")
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%")

### Plot model stats over time

In [ ]:
# Model stats over time
for stat in model_history:
    print(f"\n{stat}: \n{model_history[stat]}")

# plot loss and accuracy per epoch
plt.figure(figsize=(4, 3))
plt.title("Loss per epoch")
actual_epochs = len(model_history["train_loss"])
plt.plot(np.arange(1, actual_epochs + 1), model_history["train_loss"], label="Train Loss")
plt.plot(np.arange(1, actual_epochs + 1), model_history["val_loss"], label="Val Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.savefig("./images/loss_per_epoch.png")
plt.show()

plt.figure(figsize=(4, 3))
plt.title("Accuracy per epoch")
actual_epochs = len(model_history["train_acc"])
plt.plot(np.arange(1, actual_epochs + 1), model_history["train_acc"], label="Train Accuracy")
plt.plot(np.arange(1, actual_epochs + 1), model_history["val_acc"], label="Val Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.savefig("./images/acc_per_epoch.png")
plt.show()